## ThyNet (Thyroid Network for thyroid ultrasound image diagnosis)

### Import packages

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision
from torch.utils.tensorboard import SummaryWriter
from torchvision.models import resnet34
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import roc_auc_score, average_precision_score
from ipywidgets import FloatProgress
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from glob import glob
import os
import math
from IPython.display import display
import random
import copy
import sys

In [31]:
class Config:
    #Paths
    DATA_ROOT = "./data/" #Folder have batch1_image, batch2_image
    SPLIT_DIR = "./split/" #Folder save train/val/test split csv files
    RESULT_DIR = "./results/" #Folder save model checkpoints and logs
    LOG_DIR = RESULT_DIR + "logs/"

    #Training parameters
    BATCH_SIZE = 5  # Every time train 5 patients
    NUM_WORKERS = 4  # 12 luồng đọc ảnh → nhanh hơn
    NUM_EPOCHS = 100
    LEARNING_RATE = 1e-4  # Có thể tăng lên 5e-4 nếu cần
    # optimizer parameters
    WEIGHT_DECAY = 1e-5
    MOMENTUM = 0.9
    GAMMA = 0.1
    K_FOLDS = 5 # Chia 5 phần để train 
    TEST_SIZE = 0.1 #Tỷ lệ dữ liệu dành cho tập kiểm thử (10%). 
    SEED = 202203

    #Model parameters
    C2=128
    D=256
    #Others
    USE_TENSORBOARD = True
    RECORD_ITER=10 
config=Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


### =============== UTILITIES ====================

In [32]:
def find_batch_folders(data_root):
    """ Automatically find batch folders in the data root directory."""
    batch_folders = []
    for item in os.listdir(data_root):
        item_path = os.path.join(data_root, item)
        if os.path.isdir(item_path) and "batch" in item.lower():#dataset of batch
            # check if dataset folder and label csv files exist
            dataset_path = os.path.join(item_path, "dataset")
            label_files = glob(os.path.join(item_path, "*label*.csv"))
            if os.path.exists(dataset_path) and len(label_files) > 0:
                batch_folders.append(item)
    return sorted(batch_folders)# batch1_image,batch2_image,...

In [33]:
def collect_all_images_and_labels(data_root, batch_folders):
    """
    Collect all images and labels from multiple batch folders.
    Args:
        data_root (str): Root directory containing batch folders.
        batch_folders (list): List of batch folder names.
    Returns:
        all_images_df (pd.DataFrame): DataFrame with columns ['patient_name', 'path'].
        all_labels_df (pd.DataFrame): DataFrame with columns ['patient_name', 'histo_label'].
    """
    all_images = []
    all_labels = []
    
    for batch in batch_folders:
        print(f"\n Processing {batch}...")
        batch_path = os.path.join(data_root, batch)
        dataset_path = os.path.join(batch_path, "dataset")
        
        # Step 1:Find label file
        label_files = glob(os.path.join(batch_path, "*label*.csv"))
        if not label_files:
            print(f"No label file found in {batch}")
            continue
        label_file = label_files[0]
        
        # Step 2. Read label
        df_label = pd.read_csv(label_file)
        print(f"Loaded {len(df_label)} labels from {os.path.basename(label_file)}")
        
        # Step 3. Check required columns
        if 'patient_name' not in df_label.columns or 'histo_label' not in df_label.columns:
            print(f"Missing required columns in {batch}")
            continue

        # Step 4. Get all images
        image_paths = glob(os.path.join(dataset_path, "*.Jpg")) + \
                      glob(os.path.join(dataset_path, "*.jpg"))
        
        if not image_paths:
            print(f"No images found in {batch}/dataset/")
            continue
        
        print(f"Found {len(image_paths)} images")
        
        # Step 5. Create DataFrame of images
        df_images = pd.DataFrame({
            'patient_name': [os.path.basename(p).split('_')[0] for p in image_paths],
            'path': image_paths
        })
        
        # Step 6. Filter images with valid labels
        df_label['patient_name'] = df_label['patient_name'].astype(str)
        df_images['patient_name'] = df_images['patient_name'].astype(str)
        df_images_valid = df_images[df_images['patient_name'].isin(df_label['patient_name'])]
        
        print(f"Valid images: {len(df_images_valid)}/{len(df_images)}")
        
        all_images.append(df_images_valid)
        all_labels.append(df_label[['patient_name', 'histo_label']])
    
    # Combine all batches
    all_images_df = pd.concat(all_images, ignore_index=True)
    all_labels_df = pd.concat(all_labels, ignore_index=True).drop_duplicates('patient_name')
    
    print(f"\n{'='*60}")
    print(f"TOTAL SUMMARY:")
    print(f"Total images: {len(all_images_df)}")
    print(f"Total patients: {len(all_labels_df)}")
    print(f"Label distribution:")
    print(all_labels_df['histo_label'].value_counts())
    print(f"{'='*60}\n")
    
    return all_images_df, all_labels_df

### ================ DATASET ==================

In [34]:
class ThyDataset(Dataset):
    def __init__(self, df_images, df_labels, transform=None):
        self.df_images = df_images.reset_index(drop=True)
        self.df_labels = df_labels
        self.transform = transform
        
        # Get unique patients and sort by first occurrence
        self.unique_patients = self.df_images['patient_name'].unique()
        self.unique_patients = sorted(self.unique_patients,
            key=lambda x: self.df_images[self.df_images['patient_name'] == x].index[0])
        
    def __len__(self):
        return len(self.unique_patients)
    
    def __getitem__(self, idx):
        patient = self.unique_patients[idx]
        
        # Get label for the patient
        label_row = self.df_labels[self.df_labels['patient_name'] == patient]
        if len(label_row) == 0:
            raise ValueError(f"No label found for patient {patient}")
        label = int(label_row['histo_label'].values[0])
        
        # Get all images for the patient
        image_paths = self.df_images[self.df_images['patient_name'] == patient]['path'].tolist()
        images = []
        for img_path in image_paths:
            try:
                img = Image.open(img_path).convert('RGB')
                if self.transform:
                    img = self.transform(img)
                images.append(img)
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
                continue
        
        if len(images) == 0:
            raise ValueError(f"No valid images for patient {patient}")
        
        return torch.stack(images), label, patient

### ============= COLLATE FUNCTION ==========

In [35]:
def pad_tensor(tensor, pad, dim):
    pad_size = list(tensor.shape)
    pad_size[dim] = pad - tensor.size(dim)
    return torch.cat([tensor, torch.zeros(pad_size)], dim=dim)


In [36]:
class PadCollate:
    def __init__(self, dim=0):
        self.dim = dim
    
    def pad_collate(self, batch):
        max_len = max(map(lambda x: x[0].shape[self.dim], batch))
        images = [x[0] for x in batch]
        labels = [x[1] for x in batch]
        patients = [x[2] for x in batch]
        
        batch = map(lambda x, y, z: [pad_tensor(x, pad=max_len, dim=self.dim), y, z], 
                    images, labels, patients)
        batch = list(batch)
        
        xs = torch.stack([x[0] for x in batch], dim=0)
        ys = torch.LongTensor([x[1] for x in batch])
        zs = [x[2] for x in batch]
        return xs, ys, zs
    
    def __call__(self, batch):
        return self.pad_collate(batch)


### ==================== MODEL COMPONENTS ====================

In [ ]:
#Mô hình chú ý mối quan hệ giữa các trường hợp 
class IRAM(nn.Module):
    """Instance Relationship Attention Module"""
    def __init__(self, C1, C2=128, dropout=True, tau=128):
        super(IRAM, self).__init__()
        self.C1 = C1
        self.C2 = C2
        self.tau = tau
        
        self.psi1 = nn.Sequential(
            nn.Conv2d(C1, C2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.25) if dropout else nn.Identity()
        )
        self.psi2 = nn.Sequential(
            nn.Conv2d(C1, C2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.25) if dropout else nn.Identity()
        )
        self.psi3 = nn.Sequential(
            nn.Conv2d(C1, C2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.25) if dropout else nn.Identity()
        )
        self.psi4 = nn.Sequential(
            nn.Linear(C2, C1),
            nn.ReLU(),
            nn.Dropout(0.25) if dropout else nn.Identity()
        )
        
    def forward(self, x):
        x_psi1 = self.psi1(x).view(-1, self.C2)
        x_ds = F.interpolate(x, scale_factor=0.5)
        x_psi2 = self.psi2(x_ds).view(self.C2, -1)
        x_psi3 = self.psi3(x_ds).view(-1, self.C2)
        
        psi12 = F.softmax(torch.mm(x_psi1, x_psi2) / self.tau, dim=1)
        psi123 = torch.mm(psi12, x_psi3)
        psi1234 = self.psi4(psi123)
        M = psi1234.view(x.shape) + x
        return M, x

In [ ]:
#Mô hình số điểm ví dụ 
class ISAM(nn.Module):
    """Instance Score Attention Module"""
    def __init__(self, batch_size, L, D=256, n_classes=1, dropout=True):
        super(ISAM, self).__init__()
        self.L = L
        self.D = D
        self.batch_size = batch_size
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.attention_a = nn.Sequential(
            nn.Linear(L, D),
            nn.Tanh(),
            nn.Dropout(0.25) if dropout else nn.Identity()
        )
        self.attention_b = nn.Sequential(
            nn.Linear(L, D),
            nn.Sigmoid(),
            nn.Dropout(0.25) if dropout else nn.Identity()
        )
        self.attention_c = nn.Linear(D, n_classes)
        
    def forward(self, x):
        x_pooled = self.pool(x).view(x.shape[0], self.L)
        a = self.attention_a(x_pooled)
        b = self.attention_b(x_pooled)
        A = a.mul(b)
        A = self.attention_c(A)
        
        if not self.training:
            A = A.view(1, 1, -1)
        else:
            A = A.view(self.batch_size, 1, -1)
        return A, x_pooled, x

In [39]:
class ThyNet(nn.Module):
    """ThyNet Model"""
    def __init__(self, C2, D, batch_size, dropout=True, num_cls=2, n_channels=3):
        super(ThyNet, self).__init__()
        feature_extractor = resnet34(pretrained=True)
        self.feature_extractor = nn.Sequential(*list(feature_extractor.children())[:-2])
        
        self.C1 = self.feature_extractor[-1][-1].conv2.out_channels
        self.L = self.C1
        self.iram = IRAM(C1=self.C1, C2=C2, dropout=dropout, tau=C2)
        self.isam = ISAM(batch_size=batch_size, L=self.L, D=D, dropout=dropout)
        self.classifier = nn.Linear(self.C1, num_cls)
        
        self.n_channels = n_channels
        self.batch_size = batch_size
        
    def forward(self, x):
        x = x.view(-1, self.n_channels, x.shape[-2], x.shape[-1])
        feat = self.feature_extractor(x)
        M, _ = self.iram(feat)
        A, x_pooled, _ = self.isam(M)
        A = F.softmax(A, dim=2)
        
        if not self.training:
            x_pooled = x_pooled.view(1, -1, self.C1)
            h = torch.bmm(A, x_pooled).view(1, self.C1)
        else:
            x_pooled = x_pooled.view(self.batch_size, -1, self.C1)
            h = torch.bmm(A, x_pooled).view(self.batch_size, self.C1)
        
        logits = self.classifier(h)
        return logits

### ==================== DATA AUGMENTATION ====================

In [40]:
class Cutout(object):
    def __init__(self, n_holes, length):
        self.n_holes = n_holes
        self.length = length
    
    def __call__(self, img):
        h, w = img.size(1), img.size(2)
        mask = np.ones((h, w), np.float32)
        
        for n in range(self.n_holes):
            y = np.random.randint(h)
            x = np.random.randint(w)
            y1 = np.clip(y - self.length // 2, 0, h)
            y2 = np.clip(y + self.length // 2, 0, h)
            x1 = np.clip(x - self.length // 2, 0, w)
            x2 = np.clip(x + self.length // 2, 0, w)
            mask[y1:y2, x1:x2] = 0.
        
        mask = torch.from_numpy(mask)
        mask = mask.expand_as(img)
        img = img * mask
        return img

In [41]:
class MyRotationTrans:
    def __init__(self, angles):
        self.angles = angles
    
    def __call__(self, x):
        angle = random.choice(self.angles)
        return torchvision.transforms.functional.rotate(x, angle)

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    MyRotationTrans([0, 90, 180, 270]),
    transforms.ColorJitter(),
    transforms.ToTensor(),
    Cutout(1, 100),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### ==================== TRAINING UTILITIES ====================

In [42]:
def warmup_lr_scheduler(optimizer, warmup_iters, warmup_factor):
    def f(x):
        if x >= warmup_iters:
            return 1
        alpha = float(x) / warmup_iters
        return warmup_factor * (1 - alpha) + alpha
    return torch.optim.lr_scheduler.LambdaLR(optimizer, f)

In [43]:
@torch.no_grad()
def eval_process(epoch, model, criterion, dataloader, device):
    print(f"Epoch {epoch} Validation...")
    model.eval()
    labels, preds, running_loss = [], [], 0.
    
    for images, targets, _ in tqdm(dataloader, desc="Validating"):
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        preds.extend(logits.cpu().numpy()[:, -1].tolist())
        labels.extend(targets.tolist())
        loss = criterion(logits, targets)
        running_loss += loss.item() * images.size(0)
    
    eval_loss = running_loss / len(dataloader.dataset)
    auc_score = roc_auc_score(labels, preds)
    print(f"Val Loss: {eval_loss:.4f}, Val AUC: {auc_score:.4f}")
    return auc_score, eval_loss

In [44]:
@torch.no_grad()
def test_process(model, criterion, dataloader, device):
    model.eval()
    labels, preds_logits, preds_probs = [], [], []
    running_loss, all_patient_names = 0., []
    
    for images, targets, patient_names in tqdm(dataloader, desc="Testing"):
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        pred_probs = torch.softmax(logits, 1)
        
        preds_probs.extend(pred_probs.cpu().numpy()[:, -1].tolist())
        preds_logits.extend(logits.cpu().numpy()[:, -1].tolist())
        labels.extend(targets.tolist())
        all_patient_names.extend(patient_names)
        
        loss = criterion(logits, targets)
        running_loss += loss.item() * images.size(0)
    
    test_loss = running_loss / len(dataloader.dataset)
    auc_score = roc_auc_score(labels, preds_logits)
    ap_score = average_precision_score(labels, preds_logits)
    
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test AUC: {auc_score:.4f}")
    print(f"Test AP: {ap_score:.4f}")
    
    df = pd.DataFrame({
        "patient_name": all_patient_names,
        "prob": preds_probs,
        "logit": preds_logits,
        "label": labels
    })
    return auc_score, test_loss, df

In [45]:
def train_process(model, criterion, optimizer, lr_sche, dataloaders,
                  num_epochs, use_tensorboard, device,
                  save_model_path, record_iter, writer=None):
    model.train()
    best_score = 0.0
    best_state_dict = copy.deepcopy(model.state_dict())
    
    for epoch in range(num_epochs):
        lr_scheduler = None
        running_loss = 0.0
        print(f"\n{'='*60}")
        print(f"Epoch {epoch}/{num_epochs-1}")
        print(f"{'='*60}")
        
        if epoch == 0:
            warmup_factor = 1. / 1000
            warmup_iters = min(1000, len(dataloaders["train"]) - 1)
            lr_scheduler = warmup_lr_scheduler(optimizer, warmup_iters, warmup_factor)
        
        for i, (images, targets, _) in enumerate(tqdm(dataloaders["train"], desc="Training")):
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, targets)
            
            if not math.isfinite(loss.item()):
                print(f"Loss is {loss.item()}, stopping training")
                sys.exit(1)
            
            loss.backward()
            optimizer.step()
            if lr_scheduler is not None:
                lr_scheduler.step()
            
            running_loss += loss.item() * images.size(0)
            
            if (i + 1) % record_iter == 0:
                tmp_loss = running_loss / ((i + 1) * images.size(0))
                print(f"  Epoch {epoch} iter {i+1}: loss = {tmp_loss:.4f}")
                
                if use_tensorboard:
                    writer.add_scalar("Train loss", tmp_loss, 
                                    epoch * len(dataloaders["train"]) + i)
                    writer.add_scalar("lr", optimizer.param_groups[0]["lr"],
                                    epoch * len(dataloaders["train"]) + i)
        
        val_auc, val_loss = eval_process(epoch, model, criterion, 
                                         dataloaders["val"], device)
        
        if lr_sche is not None:
            lr_sche.step()
        
        if val_auc > best_score:
            best_score = val_auc
            best_state_dict = copy.deepcopy(model.state_dict())
            print(f"  ✓ New best AUC: {best_score:.4f}")
        
        if use_tensorboard:
            writer.add_scalar("validation AUC", val_auc, global_step=epoch)
            writer.add_scalar("validation loss", val_loss, global_step=epoch)
        
        model.train()
    
    print(f"\n{'='*60}")
    print(f"Training Done! Best Valid AUC: {best_score:.4f}")
    print(f"{'='*60}\n")
    torch.save(best_state_dict, save_model_path)
    
    print("Starting Testing...")
    model.load_state_dict(best_state_dict)
    test_auc, test_loss, df = test_process(model, criterion, 
                                           dataloaders["test"], device)
    
    if use_tensorboard:
        writer.add_scalar("Test AUC", test_auc, global_step=0)
        writer.close()
    
    return df

### ==================== MAIN TRAINING ====================

In [46]:
def split_data(all_images_df, all_labels_df, test_size, seed, k, split_dir):
    """Chia dữ liệu thành train/val/test và lưu vào split_dir"""
    os.makedirs(split_dir, exist_ok=True)
    
    # Chia test set
    train_val_labels, test_labels = train_test_split(
        all_labels_df, test_size=test_size, random_state=seed
    )
    
    test_labels.to_csv(os.path.join(split_dir, "test_case.csv"), index=False)
    test_images = all_images_df[all_images_df['patient_name'].isin(test_labels['patient_name'])]
    test_images.to_csv(os.path.join(split_dir, "test_image.csv"), index=False)
    
    # K-Fold split
    kf = KFold(n_splits=k, random_state=seed, shuffle=True)
    for i, (train_idx, val_idx) in enumerate(kf.split(train_val_labels)):
        train_labels = train_val_labels.iloc[train_idx]
        val_labels = train_val_labels.iloc[val_idx]
        
        train_labels.to_csv(os.path.join(split_dir, f"train_case_split_{i}.csv"), index=False)
        val_labels.to_csv(os.path.join(split_dir, f"val_case_split_{i}.csv"), index=False)
        
        train_images = all_images_df[all_images_df['patient_name'].isin(train_labels['patient_name'])]
        val_images = all_images_df[all_images_df['patient_name'].isin(val_labels['patient_name'])]
        
        train_images.to_csv(os.path.join(split_dir, f"train_image_split_{i}.csv"), index=False)
        val_images.to_csv(os.path.join(split_dir, f"val_image_split_{i}.csv"), index=False)
    
    print(f"✓ Split data saved to {split_dir}")

In [47]:
def train_k_fold(all_images_df, all_labels_df, config):
    """K-fold cross validation training"""
    
    # Clear GPU cache trước khi bắt đầu
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # Test dataset
    test_images = pd.read_csv(os.path.join(config.SPLIT_DIR, "test_image.csv"))
    test_labels = pd.read_csv(os.path.join(config.SPLIT_DIR, "test_case.csv"))
    test_dataset = ThyDataset(test_images, test_labels, val_transform)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, 
                            num_workers=config.NUM_WORKERS)
    
    all_results = []
    
    for fold in range(config.K_FOLDS):
        print(f"\n{'#'*60}")
        print(f"# FOLD {fold + 1}/{config.K_FOLDS}")
        print(f"{'#'*60}\n")
        
        # Load fold data
        train_images = pd.read_csv(os.path.join(config.SPLIT_DIR, f"train_image_split_{fold}.csv"))
        train_labels = pd.read_csv(os.path.join(config.SPLIT_DIR, f"train_case_split_{fold}.csv"))
        val_images = pd.read_csv(os.path.join(config.SPLIT_DIR, f"val_image_split_{fold}.csv"))
        val_labels = pd.read_csv(os.path.join(config.SPLIT_DIR, f"val_case_split_{fold}.csv"))
        
        # Create datasets
        train_dataset = ThyDataset(train_images, train_labels, train_transform)
        val_dataset = ThyDataset(val_images, val_labels, val_transform)
        
        train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,
                                 shuffle=True, collate_fn=PadCollate(dim=0),
                                 num_workers=config.NUM_WORKERS, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False,
                               num_workers=config.NUM_WORKERS)
        
        dataloaders = {"train": train_loader, "val": val_loader, "test": test_loader}
        
        # Model
        model = ThyNet(C2=config.C2, D=config.D, batch_size=config.BATCH_SIZE).to(device)
        
        # Optimizer
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.SGD(params, lr=config.LEARNING_RATE, 
                                   momentum=config.MOMENTUM,
                                   weight_decay=config.WEIGHT_DECAY)
        lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(
            optimizer, milestones=[50, 75], gamma=config.GAMMA
        )
        criterion = nn.CrossEntropyLoss()
        
        # Training
        os.makedirs(config.LOG_DIR, exist_ok=True)
        writer = SummaryWriter(os.path.join(config.LOG_DIR, f"fold_{fold+1}"))
        os.makedirs(config.RESULT_DIR, exist_ok=True)
        save_model_path = os.path.join(config.RESULT_DIR, f"thynet_fold_{fold+1}.pth")
        
        df_result = train_process(
            model=model, criterion=criterion, optimizer=optimizer,
            lr_sche=lr_scheduler, dataloaders=dataloaders,
            num_epochs=config.NUM_EPOCHS, use_tensorboard=config.USE_TENSORBOARD,
            device=device, save_model_path=save_model_path,
            record_iter=config.RECORD_ITER, writer=writer
        )
        
        df_result['fold'] = fold + 1
        all_results.append(df_result)
        
        # Clear cache sau mỗi fold
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    # Save all results
    final_results = pd.concat(all_results, ignore_index=True)
    final_results.to_csv(os.path.join(config.RESULT_DIR, "all_folds_results.csv"), index=False)
    print(f"\n✓ All results saved to {config.RESULT_DIR}/all_folds_results.csv")


### ==================== MAIN ====================

In [ ]:
if __name__ == "__main__":
    print("="*60)
    print("ThyNet Training - Auto Batch Processing")
    print("="*60)
    
    # 1. Find batch folders
    batch_folders = find_batch_folders(config.DATA_ROOT)
    if not batch_folders:
        print(f"No batch folders found in {config.DATA_ROOT}")
        sys.exit(1)
    
    print(f"\n Found {len(batch_folders)} batch folder(s): {batch_folders}")
    
    # 2. Collect all images and labels
    all_images_df, all_labels_df = collect_all_images_and_labels(
        config.DATA_ROOT, batch_folders
    )
    
    # 3. Save combined data
    os.makedirs("./data/combined", exist_ok=True)
    all_images_df.to_csv("./data/combined/all_images.csv", index=False)
    all_labels_df.to_csv("./data/combined/all_labels.csv", index=False)
    print("Saved combined data to ./data/combined/")
    
    # 4. Split data
    print("\n" + "="*60)
    print("Splitting data...")
    print("="*60)
    split_data(all_images_df, all_labels_df, 
              test_size=config.TEST_SIZE, 
              seed=config.SEED,
              k=config.0,
              split_dir=config.SPLIT_DIR)
    
    # 5. Training
    print("\n" + "="*60)
    print("Starting K-Fold Training...")
    print("="*60)
    train_k_fold(all_images_df, all_labels_df, config)
    
    print("\n" + "="*60)
    print("✓ ALL DONE!")
    print("="*60)

ThyNet Training - Auto Batch Processing

 Found 2 batch folder(s): ['batch1_image', 'batch2_image']

 Processing batch1_image...
Loaded 601 labels from batch1_image_label.csv
Found 6005 images
Valid images: 6005/6005

 Processing batch2_image...
Loaded 241 labels from batch2_image_label.csv
Found 2503 images
Valid images: 2495/2503

TOTAL SUMMARY:
Total images: 8500
Total patients: 601
Label distribution:
histo_label
1    383
0    218
Name: count, dtype: int64

Saved combined data to ./data/combined/

Splitting data...
✓ Split data saved to ./split/

Starting K-Fold Training...

############################################################
# FOLD 1/5
############################################################



/home/khanh247/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/khanh247/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Epoch 0/99


Training:   0%|          | 0/432 [00:00<?, ?it/s]

  Epoch 0 iter 10: loss = 0.7953
  Epoch 0 iter 20: loss = 0.7512
  Epoch 0 iter 30: loss = 0.7384
  Epoch 0 iter 40: loss = 0.7303
  Epoch 0 iter 50: loss = 0.7312
  Epoch 0 iter 60: loss = 0.7270
  Epoch 0 iter 70: loss = 0.7253
  Epoch 0 iter 80: loss = 0.7243
  Epoch 0 iter 90: loss = 0.7275
  Epoch 0 iter 100: loss = 0.7278
  Epoch 0 iter 110: loss = 0.7179
  Epoch 0 iter 120: loss = 0.7033
  Epoch 0 iter 130: loss = 0.6895
  Epoch 0 iter 140: loss = 0.7100
  Epoch 0 iter 150: loss = 0.7150
  Epoch 0 iter 160: loss = 0.7135
  Epoch 0 iter 170: loss = 0.7174
  Epoch 0 iter 180: loss = 0.7137
  Epoch 0 iter 190: loss = 0.7070
  Epoch 0 iter 200: loss = 0.6987
  Epoch 0 iter 210: loss = 0.6895
  Epoch 0 iter 220: loss = 0.6940
  Epoch 0 iter 230: loss = 0.6992
  Epoch 0 iter 240: loss = 0.6987
  Epoch 0 iter 250: loss = 0.7014
  Epoch 0 iter 260: loss = 0.7005
  Epoch 0 iter 270: loss = 0.6976
  Epoch 0 iter 280: loss = 0.7012
  Epoch 0 iter 290: loss = 0.7006
  Epoch 0 iter 300: los

...

  Epoch 99 iter 10: loss = 0.0629
  Epoch 99 iter 20: loss = 0.0417
  Epoch 99 iter 30: loss = 0.0409
  Epoch 99 iter 40: loss = 0.0441
  Epoch 99 iter 50: loss = 0.0421
  Epoch 99 iter 60: loss = 0.0411
  Epoch 99 iter 70: loss = 0.0518
  Epoch 99 iter 80: loss = 0.0512
  Epoch 99 iter 90: loss = 0.0625
  Epoch 99 iter 100: loss = 0.0646
  Epoch 99 iter 110: loss = 0.0662
  Epoch 99 iter 120: loss = 0.0633
  Epoch 99 iter 130: loss = 0.0593
  Epoch 99 iter 140: loss = 0.0586
  Epoch 99 iter 150: loss = 0.0558
  Epoch 99 iter 160: loss = 0.0537
  Epoch 99 iter 170: loss = 0.0520
  Epoch 99 iter 180: loss = 0.0508
  Epoch 99 iter 190: loss = 0.0511
  Epoch 99 iter 200: loss = 0.0507
  Epoch 99 iter 210: loss = 0.0497
  Epoch 99 iter 220: loss = 0.0527
  Epoch 99 iter 230: loss = 0.0543
  Epoch 99 iter 240: loss = 0.0614
  Epoch 99 iter 250: loss = 0.0617
  Epoch 99 iter 260: loss = 0.0609
  Epoch 99 iter 270: loss = 0.0595
  Epoch 99 iter 280: loss = 0.0606
  Epoch 99 iter 290: loss = 0

Validating:   0%|          | 0/108 [00:00<?, ?it/s]

Val Loss: 1.6027, Val AUC: 0.4504

Training Done! Best Valid AUC: 0.5385

Starting Testing...


Testing:   0%|          | 0/61 [00:00<?, ?it/s]

Test Loss: 0.9317
Test AUC: 0.4598
Test AP: 0.7027

✓ All results saved to ./results//all_folds_results.csv

✓ ALL DONE!
